In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from datasets import Dataset
from sklearn.metrics import classification_report

### Load Model Llama-3.1-8B-Instruct

In [ ]:
# pip install -U "transformers>=4.43" "peft>=0.11.0" accelerate datasets sentencepiece
# Optional: pip install flash-attn --no-build-isolation  (if your CUDA setup supports it)

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, DataCollatorForLanguageModeling, Trainer
)
from peft import LoraConfig, get_peft_model, TaskType

base_model = "meta-llama/Llama-3.1-8B-Instruct"  # or the 8B base if you prefer
tokenizer = AutoTokenizer.from_pretrained(base_model, use_fast=False)
# Llama tokenizers usually need this set for padding:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load in bf16 to keep memory reasonable (A100/MI300/BF16-capable GPU)
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="cuda"
)


2025-09-30 16:45:29.532214: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-30 16:45:29.543372: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1759265129.557234 1226140 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1759265129.561268 1226140 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-09-30 16:45:29.575316: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

### DoRa

In [3]:
# DoRA via PEFT: enable with use_dora=True
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    use_dora=True,
    # Llama 3.x target modules:
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()  # sanity check

trainable params: 22,347,776 || all params: 8,052,609,024 || trainable%: 0.2775


### HateXplain

In [17]:
hx = pd.read_csv('../Hate Speech/hatexplain_processed.csv')
hx.head()

,id,annotators,rationales,post_tokens,text,majority_label,class_label
0,18790322_gab,"{'label': [2, 1, 1], 'annotator_id': [203, 202...",[],"['i', 'live', 'and', 'work', 'with', 'many', '...",i live and work with many legal mexican immigr...,0,Normal
1,13459307_gab,"{'label': [0, 2, 0], 'annotator_id': [67, 57, ...","[[0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0,...","['in', 'britain', 'the', 'prime', 'minister', ...",in britain the prime minister a childless woma...,1,Hate
2,1113705567782969344_twitter,"{'label': [1, 2, 1], 'annotator_id': [207, 221...",[],"['<user>', '<user>', 'there', 'shd', 'be', 'a'...",<user> <user> there shd be a film called no hi...,0,Normal
3,24555628_gab,"{'label': [2, 1, 1], 'annotator_id': [246, 221...",[],"['jews', 'defends', 'jews', 'regardless', 'mus...",jews defends jews regardless muslims defend mu...,0,Normal
4,16382456_gab,"{'label': [0, 0, 0], 'annotator_id': [228, 220...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,...","['the', 'non', 'partisan', 'congressional', 'b...",the non partisan congressional budget office c...,1,Hate


In [5]:
from datasets import Dataset

def preprocess_dataset(df, text_col, label_col, tokenizer, max_len=2048, split_ratio=0.2, seed=42):
    """
    Prepare a dataset for instruction-tuning with text labels.
    Each row becomes: <prompt> + <label_text> for causal LM training.
    
    Args:
        df (pd.DataFrame): input dataframe
        text_col (str): column name containing the input text
        label_col (str): column name containing the target label (string, e.g. "Hate")
        tokenizer: Hugging Face tokenizer
        max_len (int): max sequence length
        split_ratio (float): fraction to use for eval set (default 0.2)
        seed (int): random seed for reproducibility
    """
    ds = Dataset.from_pandas(df)

    def format_example(ex):
        # Build an instruction-style prompt
        prompt = (
        "You are a language model trained to evaluate social media posts and determine "
        "whether the content constitutes Hate Speech or Not Hate Speech based on a provided definition.\n\n"
        "Definition of Hate Speech:\n"
        "Hate speech refers to any language that attacks, diminishes, incites violence against, "
        "or promotes hatred toward individuals or groups based on characteristics such as physical appearance, "
        "religion, descent, national or ethnic origin, sexual orientation, gender identity, or other inherent attributes. "
        "Hate speech may be conveyed overtly or subtly, and can include sarcastic, humorous, or coded expressions "
        "intended to convey hostility or exclusion.\n\n"
        f"Post: {ex[text_col]}\n"
        "Your task is to analyze the post and then return a JSON object:\n"
        "{\n"
        '  "label": "Hate" | "Not Hate"\n'
        "}"
        )

        label_text = ex[label_col]

        # Full sequence = prompt + answer
        full_text = prompt + " " + label_text

        tokenized = tokenizer(
            full_text,
            truncation=True,
            max_length=max_len,
            padding="max_length"
        )
        return tokenized

    tokenized = ds.map(format_example, batched=False, remove_columns=ds.column_names)
    split_ds = tokenized.train_test_split(test_size=split_ratio, seed=seed)
    return split_ds["train"], split_ds["test"]


In [6]:
train_ds, eval_ds = preprocess_dataset(
    hx,
    text_col="text",
    label_col="class_label",   # e.g., contains "Hate" or "Not Hate"
    tokenizer=tokenizer
)
print(train_ds[0])


Map:   0%|          | 0/10999 [00:00<?, ? examples/s]

{'input_ids': [128000, 2675, 527, 264, 4221, 1646, 16572, 311, 15806, 3674, 3772, 8158, 323, 8417, 3508, 279, 2262, 42675, 66912, 39841, 477, 2876, 66912, 39841, 3196, 389, 264, 3984, 7419, 382, 10614, 315, 66912, 39841, 512, 39, 349, 8982, 19813, 311, 904, 4221, 430, 8951, 11, 57160, 288, 11, 3709, 3695, 9349, 2403, 11, 477, 39990, 35242, 9017, 7931, 477, 5315, 3196, 389, 17910, 1778, 439, 7106, 11341, 11, 13901, 11, 38052, 11, 5426, 477, 22277, 6371, 11, 7392, 17140, 11, 10026, 9764, 11, 477, 1023, 38088, 8365, 13, 66912, 8982, 1253, 387, 73897, 43661, 398, 477, 87417, 11, 323, 649, 2997, 83367, 292, 11, 70946, 11, 477, 47773, 24282, 10825, 311, 20599, 61029, 477, 42308, 382, 4226, 25, 912, 10712, 922, 433, 374, 24705, 374, 311, 19065, 1475, 892, 1063, 81027, 114926, 18071, 8349, 568, 477, 1364, 1550, 433, 369, 1063, 1912, 433, 374, 311, 84040, 279, 19065, 505, 279, 1972, 75283, 374, 24705, 41040, 387, 5304, 433, 198, 7927, 3465, 374, 311, 24564, 279, 1772, 323, 1243, 471, 264, 4823,

In [7]:
small_train = train_ds.select(range(32))   # first 32 samples
small_eval  = eval_ds.select(range(16))

In [9]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def compute_metrics(eval_pred):
    # eval_pred is (logits, labels) for classification
    # but for CausalLM SFT it may be raw predictions → need postprocessing
    predictions, labels = eval_pred

    # Convert token IDs to strings
    pred_str = tokenizer.batch_decode(np.argmax(predictions, axis=-1), skip_special_tokens=True)
    label_str = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # Clean up whitespace
    pred_str = [p.strip() for p in pred_str]
    label_str = [l.strip() for l in label_str]

    # Map to binary
    y_true = [1 if "Hate" in l else 0 for l in label_str]
    y_pred = [1 if "Hate" in p else 0 for p in pred_str]

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred)
    }


In [10]:
from trl import SFTTrainer, SFTConfig

sft_args = SFTConfig(
    output_dir="llama31-8b-dora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=25,
    bf16=True,    # or fp16=True if not supported
    save_steps=200,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=200,
    gradient_checkpointing=True,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    report_to="tensorboard",
    max_length=2048,   # keep under LLaMA’s context size
)

trainer = SFTTrainer(
    model=model,
    train_dataset=small_train,
    eval_dataset=small_eval,  # column containing your combined prompt+label text
    args=sft_args,
    compute_metrics=compute_metrics,
)

trainer.train()


Truncating train dataset:   0%|          | 0/32 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/16 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss


TrainOutput(global_step=6, training_loss=0.17176955938339233, metrics={'train_runtime': 400.4138, 'train_samples_per_second': 0.24, 'train_steps_per_second': 0.015, 'total_flos': 8879531888738304.0, 'train_loss': 0.17176955938339233, 'entropy': 0.17489964701235294, 'num_tokens': 196608.0, 'mean_token_accuracy': 0.9659104868769646, 'epoch': 3.0})

In [19]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
import torch
from tqdm import tqdm

def evaluate_on_csv(model, tokenizer, df, text_col, label_col, max_new_tokens=20, n_samples=100, device="cuda"):

    # sample smaller subset for quick test
    if n_samples and n_samples < len(df):
        df = df.sample(n=n_samples, random_state=42)

    prompts = [f"Classify the following post as Hate or Not Hate:\nPost: {t}\nLabel:" for t in df[text_col].tolist()]
    gold_labels = df[label_col].tolist()

    model.eval().to(device)

    preds = []
    for i in tqdm(range(0, len(prompts), 4)):  # batch size 4
        batch_prompts = prompts[i:i+4]
        inputs = tokenizer(batch_prompts, return_tensors="pt", padding=True).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False
            )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        preds.extend([d.split("Label:")[-1].strip() for d in decoded])

    # Convert to binary
    y_true = [1 if "Hate" in l else 0 for l in gold_labels]
    y_pred = [1 if "Hate" in p else 0 for p in preds]

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")

    return acc, f1, preds, gold_labels, df[text_col].tolist()


In [20]:
acc, f1, preds, gold, posts = evaluate_on_csv(
    model, tokenizer,
    hx,
    text_col="text",       # your input text column
    label_col="class_label", # your gold label col
    n_samples=50           # quick test on 50 samples
)


  0%|          | 0/13 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


  8%|▊         | 1/13 [00:12<02:24, 12.05s/it]Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
 15%|█▌        | 2/13 [00:24<02:12, 12.04s/it]Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
 23%|██▎       | 3/13 [00:36<02:00, 12.04s/it]Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
 31%|███       | 4/13 [00:48<01:48, 12.09s/it]Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.
A dec

Accuracy: 0.4800
F1 Score: 0.6486


In [ ]:
# Save the DoRA adapter (small files)
model.save_pretrained("llama31-8b-dora-adapter")
tokenizer.save_pretrained("llama31-8b-dora-adapter")

# Optional: merge adapter for single-file inference (no extra runtime cost)
merged = model.merge_and_unload()
merged.save_pretrained("llama31-8b-dora-merged")
tokenizer.save_pretrained("llama31-8b-dora-merged")